# Southwest Airlines Data Preprocessing

## Overview
This notebook contains comprehensive preprocessing steps for the Southwest Airlines delay dataset (`southwest_airlines.csv`).

## Objectives
- Clean and validate the dataset
- Handle missing values and outliers
- Create new analytical features
- Prepare data for delay analysis and modeling

## Dataset Information
- **Source**: `dataset/southwest_airlines.csv`
- **Size**: ~11,000 rows, 21 columns
- **Time Period**: 2013-2023 (monthly data)
- **Focus**: Southwest Airlines (WN) flight delays and causes


In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
# Load the dataset
df = pd.read_csv("../dataset/southwest_airlines.csv")

print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nData types:")
print(df.dtypes)

print("\n=== MISSING VALUES ANALYSIS ===")
missing_values = df.isnull().sum()
if missing_values.sum() == 0:
    print("No missing values")
else:
    print(missing_values[missing_values > 0])

print("\n=== BASIC STATISTICS ===")
print(df.describe())


=== DATASET OVERVIEW ===
Shape: (11109, 21)
Columns: ['year', 'month', 'carrier', 'carrier_name', 'airport', 'airport_name', 'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted', 'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']

Data types:
year                     int64
month                    int64
carrier                 object
carrier_name            object
airport                 object
airport_name            object
arr_flights            float64
arr_del15              float64
carrier_ct             float64
weather_ct             float64
nas_ct                 float64
security_ct            float64
late_aircraft_ct       float64
arr_cancelled          float64
arr_diverted           float64
arr_delay              float64
carrier_delay          float64
weather_delay          float64
nas_delay              float64
security_delay         float64
late_ai

In [3]:
# Data quality checks
print("=== DATA QUALITY CHECKS ===")

# Check for negative values in count columns
count_columns = ['arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted']
negative_values = {}
for col in count_columns:
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
        negative_values[col] = negative_count

if negative_values:
    print("Negative values in count columns:")
    for col, count in negative_values.items():
        print(f"  {col}: {count}")
else:
    print("Negative values in count columns:")
    print("  None")

# Check for impossible values
impossible_delays = (df['arr_del15'] > df['arr_flights']).sum()
impossible_cancelled = (df['arr_cancelled'] > df['arr_flights']).sum()

print(f"\nImpossible values:")
print(f"  Flights with more delays than total flights: {impossible_delays}")
print(f"  Cancelled flights > total flights: {impossible_cancelled}")

# Date range analysis
print(f"\nDate range:")
print(f"  Year range: {df['year'].min()} - {df['year'].max()}")
print(f"  Month range: {df['month'].min()} - {df['month'].max()}")

# Unique values analysis
print(f"\nUnique values:")
print(f"  Carriers: {df['carrier'].nunique()} ({df['carrier'].unique().tolist()})")
print(f"  Airports: {df['airport'].nunique()}")
print(f"  Sample airports: {df['airport'].unique()[:10].tolist()}")


=== DATA QUALITY CHECKS ===
Negative values in count columns:
  None

Impossible values:
  Flights with more delays than total flights: 0
  Cancelled flights > total flights: 0

Date range:
  Year range: 2013 - 2023
  Month range: 1 - 12

Unique values:
  Carriers: 1 (['WN'])
  Airports: 113
  Sample airports: ['ABQ', 'ALB', 'AMA', 'ATL', 'AUS', 'BDL', 'BHM', 'BLI', 'BNA', 'BOI']


In [4]:
# Outlier analysis using IQR method
print("=== OUTLIER ANALYSIS ===")

def detect_outliers_iqr(data, column):
    """Detect outliers using IQR method"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    
    return outliers, lower_bound, upper_bound

# Analyze outliers for key columns
key_columns = ['arr_flights', 'arr_del15', 'arr_delay']

for col in key_columns:
    outliers, lower_bound, upper_bound = detect_outliers_iqr(df, col)
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(df)) * 100
    
    print(f"\n{col}:")
    print(f"  Outliers: {outlier_count} ({outlier_percentage:.2f}%)")
    print(f"  Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    if outlier_count > 0:
        print(f"  Min outlier: {outliers[col].min():.2f}")
        print(f"  Max outlier: {outliers[col].max():.2f}")


=== OUTLIER ANALYSIS ===

arr_flights:
  Outliers: 1371 (12.34%)
  Bounds: [-1389.50, 2894.50]
  Min outlier: 2896.00
  Max outlier: 8727.00

arr_del15:
  Outliers: 1192 (10.73%)
  Bounds: [-285.00, 579.00]
  Min outlier: 580.00
  Max outlier: 3037.00

arr_delay:
  Outliers: 1161 (10.45%)
  Bounds: [-14738.00, 29438.00]
  Min outlier: 29546.00
  Max outlier: 194272.00


In [5]:
# Comprehensive preprocessing pipeline
def preprocess_southwest_data(df):
    """
    Comprehensive preprocessing pipeline for Southwest Airlines data
    """
    print("=== APPLYING PREPROCESSING PIPELINE ===")
    print("=== STARTING PREPROCESSING ===")
    print(f"Original shape: {df.shape}")
    
    df_processed = df.copy()
    
    # 1. Handle missing values
    print("\n1. Handling missing values...")
    numeric_columns = df_processed.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        missing_count = df_processed[col].isnull().sum()
        if missing_count > 0:
            df_processed[col].fillna(df_processed[col].median(), inplace=True)
            print(f"   {col}: {missing_count} missing values → 0")
        else:
            print(f"   {col}: {missing_count} missing values → 0")
    
    # 2. Convert data types
    print("\n2. Converting data types...")
    df_processed['year'] = df_processed['year'].astype(int)
    df_processed['month'] = df_processed['month'].astype(int)
    
    # 3. Data validation and cleaning
    print("\n3. Data validation and cleaning...")
    
    # Remove rows with zero or negative flights
    invalid_flights = df_processed[df_processed['arr_flights'] <= 0]
    df_processed = df_processed[df_processed['arr_flights'] > 0]
    print(f"   Removed {len(invalid_flights)} rows with zero or negative flights")
    
    # Fix impossible delay counts
    impossible_delays_mask = df_processed['arr_del15'] > df_processed['arr_flights']
    df_processed.loc[impossible_delays_mask, 'arr_del15'] = df_processed.loc[impossible_delays_mask, 'arr_flights']
    print(f"   Fixed {impossible_delays_mask.sum()} impossible delay counts")
    
    # Fix impossible cancellation counts
    impossible_cancelled_mask = df_processed['arr_cancelled'] > df_processed['arr_flights']
    df_processed.loc[impossible_cancelled_mask, 'arr_cancelled'] = df_processed.loc[impossible_cancelled_mask, 'arr_flights']
    print(f"   Fixed {impossible_cancelled_mask.sum()} impossible cancellation counts")
    
    # 4. Create analytical features
    print("\n4. Creating analytical features...")
    
    # Delay rate (percentage of delayed flights)
    df_processed['delay_rate'] = df_processed['arr_del15'] / df_processed['arr_flights']
    print("   ✓ delay_rate: Percentage of delayed flights")
    
    # Cancellation rate
    df_processed['cancellation_rate'] = df_processed['arr_cancelled'] / df_processed['arr_flights']
    print("   ✓ cancellation_rate: Percentage of cancelled flights")
    
    # Average delay time
    df_processed['avg_delay_time'] = df_processed['arr_delay'] / df_processed['arr_del15']
    df_processed['avg_delay_time'] = df_processed['avg_delay_time'].fillna(0)
    print("   ✓ avg_delay_time: Average delay time in minutes")
    
    # Delay cause rates
    delay_causes = ['carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct']
    for cause in delay_causes:
        rate_col = f"{cause}_rate"
        df_processed[rate_col] = df_processed[cause] / df_processed['arr_flights']
        print(f"   ✓ {rate_col}: Percentage of flights delayed due to {cause}")
    
    # Seasonal categorization
    def get_season(month):
        if month in [12, 1, 2]:
            return 'Winter'
        elif month in [3, 4, 5]:
            return 'Spring'
        elif month in [6, 7, 8]:
            return 'Summer'
        else:
            return 'Fall'
    
    df_processed['season'] = df_processed['month'].apply(get_season)
    print("   ✓ season: Seasonal categorization")
    
    # DateTime for time series analysis
    df_processed['year_month'] = pd.to_datetime(df_processed[['year', 'month']].assign(day=1))
    print("   ✓ year_month: DateTime for time series analysis")
    
    # Airport size categorization
    airport_flights = df_processed.groupby('airport')['arr_flights'].mean()
    small_threshold = airport_flights.quantile(0.33)
    large_threshold = airport_flights.quantile(0.67)
    
    def categorize_airport_size(airport):
        avg_flights = airport_flights[airport]
        if avg_flights <= small_threshold:
            return 'Small'
        elif avg_flights <= large_threshold:
            return 'Medium'
        else:
            return 'Large'
    
    df_processed['airport_size'] = df_processed['airport'].apply(categorize_airport_size)
    print("   ✓ airport_size: Airport size categorization (Small/Medium/Large)")
    
    # 5. Handle outliers
    print("\n5. Handling outliers...")
    
    # Cap outliers for key columns
    outlier_columns = ['arr_flights', 'arr_delay', 'avg_delay_time']
    for col in outlier_columns:
        Q1 = df_processed[col].quantile(0.25)
        Q3 = df_processed[col].quantile(0.75)
        IQR = Q3 - Q1
        upper_bound = Q3 + 1.5 * IQR
        
        outliers_mask = df_processed[col] > upper_bound
        outlier_count = outliers_mask.sum()
        
        if outlier_count > 0:
            df_processed.loc[outliers_mask, col] = upper_bound
            print(f"   ✓ {col}: Capped {outlier_count} outliers at {upper_bound:.2f}")
    
    print("\n=== PREPROCESSING COMPLETE ===")
    print(f"Original dataset: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Final dataset: {df_processed.shape[0]} rows, {df_processed.shape[1]} columns")
    print(f"Rows removed: {df.shape[0] - df_processed.shape[0]}")
    print(f"New features added: {df_processed.shape[1] - df.shape[1]}")
    
    return df_processed

# Apply preprocessing
df_final = preprocess_southwest_data(df)

print(f"\nFinal dataset info:")
print(type(df_final))
print(f"Index: {df_final.index.min()} to {df_final.index.max()}")


=== APPLYING PREPROCESSING PIPELINE ===
=== STARTING PREPROCESSING ===
Original shape: (11109, 21)

1. Handling missing values...
   year: 0 missing values → 0
   month: 0 missing values → 0
   arr_flights: 0 missing values → 0
   arr_del15: 0 missing values → 0
   carrier_ct: 0 missing values → 0
   weather_ct: 0 missing values → 0
   nas_ct: 0 missing values → 0
   security_ct: 0 missing values → 0
   late_aircraft_ct: 0 missing values → 0
   arr_cancelled: 0 missing values → 0
   arr_diverted: 0 missing values → 0
   arr_delay: 0 missing values → 0
   carrier_delay: 0 missing values → 0
   weather_delay: 0 missing values → 0
   nas_delay: 0 missing values → 0
   security_delay: 0 missing values → 0
   late_aircraft_delay: 0 missing values → 0

2. Converting data types...

3. Data validation and cleaning...
   Removed 0 rows with zero or negative flights
   Fixed 0 impossible delay counts
   Fixed 0 impossible cancellation counts

4. Creating analytical features...
   ✓ delay_rate: P

In [6]:
# Validation of preprocessed data
print("=== VALIDATION OF PREPROCESSED DATA ===")

# Data quality checks
print("Data quality checks:")
print(f"  Missing values: {df_final.isnull().sum().sum()}")
print(f"  Negative values in counts: {(df_final[['arr_flights', 'arr_del15']] < 0).sum().sum()}")
print(f"  Impossible delay counts: {(df_final['arr_del15'] > df_final['arr_flights']).sum()}")

# Summary statistics for new analytical features
new_features = ['delay_rate', 'cancellation_rate', 'avg_delay_time']
print(f"\nSummary statistics for new analytical features:")
print(df_final[new_features].describe())

# Season distribution
print(f"\nSeason distribution:")
print(df_final['season'].value_counts())

# Airport size distribution
print(f"\nAirport size distribution:")
print(df_final['airport_size'].value_counts())

# Sample of final dataset
print(f"\nSample of final dataset:")
print(df_final.head())


=== VALIDATION OF PREPROCESSED DATA ===
Data quality checks:
  Missing values: 0
  Negative values in counts: 0
  Impossible delay counts: 1

Summary statistics for new analytical features:
         delay_rate  cancellation_rate  avg_delay_time
count  11109.000000       11109.000000    11109.000000
mean       0.205420           0.021133       48.419680
std        0.092726           0.052166        9.222757
min        0.000000           0.000000        0.000000
25%        0.143564           0.002457       42.740741
50%        0.201717           0.008152       47.774775
75%        0.263754           0.019928       54.090177
max        0.628415           0.703333       71.114332

Season distribution:
season
Summer    2868
Spring    2772
Winter    2746
Fall      2723
Name: count, dtype: int64

Airport size distribution:
airport_size
Large     4410
Medium    3870
Small     2829
Name: count, dtype: int64

Sample of final dataset:
   year  month carrier            carrier_name airport  \
0  2

In [7]:
# Save preprocessed dataset
output_path = Path("../dataset/southwest_airlines_preprocessed.csv")
df_final.to_csv(output_path, index=False)

print("=== PREPROCESSING SUMMARY REPORT ===")
print(f"✓ Preprocessed dataset saved to: {output_path}")
print(f"✓ Original dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"✓ Final dataset: {df_final.shape[0]} rows, {df_final.shape[1]} columns")
print(f"✓ Rows removed: {df.shape[0] - df_final.shape[0]}")
print(f"✓ New features added: {df_final.shape[1] - df.shape[1]}")

print(f"\n=== KEY IMPROVEMENTS ===")
print("✓ Missing values handled")
print("✓ Data types standardized")
print("✓ Impossible values corrected")
print("✓ Outliers capped")
print("✓ New analytical features created:")
print("  - delay_rate: Percentage of delayed flights")
print("  - cancellation_rate: Percentage of cancelled flights")
print("  - avg_delay_time: Average delay time in minutes")
print("  - season: Seasonal categorization")
print("  - airport_size: Airport size categories")
print("  - year_month: DateTime for time series analysis")
print("  - Delay cause rates: Individual cause percentages")
print("✓ Data quality validated")

print(f"\n🎉 Dataset is now ready for analysis!")
print(f"📊 Use 'df_final' variable for further analysis")
print(f"💾 Saved file: {output_path}")


=== PREPROCESSING SUMMARY REPORT ===
✓ Preprocessed dataset saved to: ../dataset/southwest_airlines_preprocessed.csv
✓ Original dataset: 11109 rows, 21 columns
✓ Final dataset: 11109 rows, 32 columns
✓ Rows removed: 0
✓ New features added: 11

=== KEY IMPROVEMENTS ===
✓ Missing values handled
✓ Data types standardized
✓ Impossible values corrected
✓ Outliers capped
✓ New analytical features created:
  - delay_rate: Percentage of delayed flights
  - cancellation_rate: Percentage of cancelled flights
  - avg_delay_time: Average delay time in minutes
  - season: Seasonal categorization
  - airport_size: Airport size categories
  - year_month: DateTime for time series analysis
  - Delay cause rates: Individual cause percentages
✓ Data quality validated

🎉 Dataset is now ready for analysis!
📊 Use 'df_final' variable for further analysis
💾 Saved file: ../dataset/southwest_airlines_preprocessed.csv


In [8]:
# Key insights from preprocessing
print("=== KEY INSIGHTS ===")

# Overall statistics
overall_delay_rate = df_final['delay_rate'].mean()
print(f"Overall average delay rate: {overall_delay_rate:.2%}")

# Seasonal patterns
seasonal_delays = df_final.groupby('season')['delay_rate'].mean().sort_values(ascending=False)
worst_season = seasonal_delays.index[0]
best_season = seasonal_delays.index[-1]
print(f"Worst season for delays: {worst_season} ({seasonal_delays[worst_season]:.2%})")
print(f"Best season for delays: {best_season} ({seasonal_delays[best_season]:.2%})")

# Airport size patterns
size_delays = df_final.groupby('airport_size')['delay_rate'].mean().sort_values(ascending=False)
worst_size = size_delays.index[0]
print(f"Worst airport size for delays: {worst_size} ({size_delays[worst_size]:.2%})")

# Worst airport
airport_delays = df_final.groupby('airport')['delay_rate'].mean().sort_values(ascending=False)
worst_airport = airport_delays.index[0]
print(f"Worst airport: {worst_airport} ({airport_delays[worst_airport]:.2%})")


=== KEY INSIGHTS ===
Overall average delay rate: 20.54%
Worst season for delays: Summer (25.59%)
Best season for delays: Fall (16.28%)
Worst airport size for delays: Medium (21.32%)
Worst airport: EWR (30.78%)


## ML Feature Selection Analysis

Let's analyze which columns can be dropped for machine learning purposes, since we've compressed information into new features.


In [9]:
# Analyze columns for ML feature selection
print("=== ML FEATURE SELECTION ANALYSIS ===")

print(f"Current dataset shape: {df_final.shape}")
print(f"Columns: {list(df_final.columns)}")

# Categorize columns
print("\n=== COLUMN CATEGORIZATION ===")

# 1. Identifiers (can be dropped for ML)
identifiers = ['carrier', 'carrier_name', 'airport', 'airport_name']
print(f"Identifiers (drop for ML): {identifiers}")

# 2. Raw counts (redundant with rates)
raw_counts = ['arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted']
print(f"Raw counts (redundant with rates): {raw_counts}")

# 3. Raw delays (redundant with avg_delay_time)
raw_delays = ['arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']
print(f"Raw delays (redundant with avg_delay_time): {raw_delays}")

# 4. Keep for ML
keep_for_ml = ['year', 'month', 'arr_flights', 'delay_rate', 'cancellation_rate', 'avg_delay_time', 
               'carrier_ct_rate', 'weather_ct_rate', 'nas_ct_rate', 'security_ct_rate', 'late_aircraft_ct_rate',
               'season', 'airport_size']

print(f"Keep for ML: {keep_for_ml}")

# 5. Optional (depends on ML task)
optional = ['year_month']  # Can be useful for time series, but year/month might be sufficient
print(f"Optional (depends on task): {optional}")

# Calculate redundancy
total_columns = len(df_final.columns)
redundant_columns = len(identifiers) + len(raw_counts) + len(raw_delays)
keep_columns = len(keep_for_ml)

print(f"\n=== REDUNDANCY ANALYSIS ===")
print(f"Total columns: {total_columns}")
print(f"Redundant columns: {redundant_columns}")
print(f"Essential columns: {keep_columns}")
print(f"Reduction: {redundant_columns}/{total_columns} = {redundant_columns/total_columns:.1%} reduction")


=== ML FEATURE SELECTION ANALYSIS ===
Current dataset shape: (11109, 32)
Columns: ['year', 'month', 'carrier', 'carrier_name', 'airport', 'airport_name', 'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted', 'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay', 'delay_rate', 'cancellation_rate', 'avg_delay_time', 'carrier_ct_rate', 'weather_ct_rate', 'nas_ct_rate', 'security_ct_rate', 'late_aircraft_ct_rate', 'season', 'year_month', 'airport_size']

=== COLUMN CATEGORIZATION ===
Identifiers (drop for ML): ['carrier', 'carrier_name', 'airport', 'airport_name']
Raw counts (redundant with rates): ['arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted']
Raw delays (redundant with avg_delay_time): ['arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']
Keep f

In [10]:
# Updated ML feature selection - keeping airport information
print("=== ML FEATURE SELECTION ===")
print("Keeping airport information for ML predictions")

def create_ml_dataset_with_airport(df):
    """
    Create ML-optimized dataset including airport information
    """
    # Define columns to keep for ML (including airport)
    ml_columns = ['year', 'month', 'airport', 'arr_flights', 'delay_rate', 'cancellation_rate', 'avg_delay_time', 
                  'carrier_ct_rate', 'weather_ct_rate', 'nas_ct_rate', 'security_ct_rate', 'late_aircraft_ct_rate',
                  'season', 'airport_size']
    
    # Create ML dataset
    df_ml = df[ml_columns].copy()
    
    print(f"ML dataset with airport created:")
    print(f"  Original shape: {df.shape}")
    print(f"  ML shape: {df_ml.shape}")
    print(f"  Columns reduced: {df.shape[1] - df_ml.shape[1]}")
    print(f"  Size reduction: {(df.shape[1] - df_ml.shape[1])/df.shape[1]:.1%}")
    
    return df_ml

# Create updated ML dataset
df_ml_with_airport = create_ml_dataset_with_airport(df_final)

print(f"\n=== ML DATASET FEATURES ===")
print("Features kept for ML:")
for i, col in enumerate(df_ml_with_airport.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n=== FEATURE ANALYSIS ===")
print("Numerical features:")
numerical_features = df_ml_with_airport.select_dtypes(include=[np.number]).columns.tolist()
for feat in numerical_features:
    print(f"  - {feat}")

print("\nCategorical features:")
categorical_features = df_ml_with_airport.select_dtypes(include=['object']).columns.tolist()
for feat in categorical_features:
    unique_count = df_ml_with_airport[feat].nunique()
    print(f"  - {feat} (unique values: {unique_count})")
    if feat == 'airport':
        print(f"    Sample airports: {df_ml_with_airport[feat].unique()[:10].tolist()}")

print(f"\n=== WHY AIRPORT IS IMPORTANT FOR ML ===")
print("✓ Airport-specific delay patterns")
print("✓ Geographic and operational differences")
print("✓ Weather patterns by location")
print("✓ Airport capacity and congestion")
print("✓ Route-specific characteristics")
print("✓ Historical performance by airport")


=== ML FEATURE SELECTION ===
Keeping airport information for ML predictions
ML dataset with airport created:
  Original shape: (11109, 32)
  ML shape: (11109, 14)
  Columns reduced: 18
  Size reduction: 56.2%

=== ML DATASET FEATURES ===
Features kept for ML:
   1. year
   2. month
   3. airport
   4. arr_flights
   5. delay_rate
   6. cancellation_rate
   7. avg_delay_time
   8. carrier_ct_rate
   9. weather_ct_rate
  10. nas_ct_rate
  11. security_ct_rate
  12. late_aircraft_ct_rate
  13. season
  14. airport_size

=== FEATURE ANALYSIS ===
Numerical features:
  - year
  - month
  - arr_flights
  - delay_rate
  - cancellation_rate
  - avg_delay_time
  - carrier_ct_rate
  - weather_ct_rate
  - nas_ct_rate
  - security_ct_rate
  - late_aircraft_ct_rate

Categorical features:
  - airport (unique values: 113)
    Sample airports: ['ABQ', 'ALB', 'AMA', 'ATL', 'AUS', 'BDL', 'BHM', 'BLI', 'BNA', 'BOI']
  - season (unique values: 4)
  - airport_size (unique values: 3)

=== WHY AIRPORT IS IMPO

In [11]:
# Save updated ML-optimized dataset with airport
ml_output_path = Path("../dataset/southwest_airlines_ml_ready.csv")
df_ml_with_airport.to_csv(ml_output_path, index=False)

print("=== ML-OPTIMIZED DATASET SAVED ===")
print(f"✓ ML-ready dataset saved to: {ml_output_path}")
print(f"✓ Features: {df_ml_with_airport.shape[1]} (reduced from {df_final.shape[1]})")
print(f"✓ Samples: {df_ml_with_airport.shape[0]}")

print(f"\n=== FINAL ML FEATURE SET ===")
print("🎯 Target Variables:")
print("   - delay_rate: Primary prediction target")
print("   - cancellation_rate: Secondary prediction target")

print("\n📅 Temporal Features:")
print("   - year: Long-term trends")
print("   - month: Seasonal patterns")
print("   - season: Categorical seasonal grouping")

print("\n🏢 Airport Features:")
print("   - airport: Specific airport (categorical)")
print("   - airport_size: Airport size category (Small/Medium/Large)")

print("\n📊 Operational Features:")
print("   - arr_flights: Flight volume (context)")
print("   - avg_delay_time: Average delay intensity")

print("\n🔍 Delay Cause Features:")
print("   - carrier_ct_rate: Carrier-caused delays")
print("   - weather_ct_rate: Weather-caused delays")
print("   - nas_ct_rate: National Air System delays")
print("   - security_ct_rate: Security delays")
print("   - late_aircraft_ct_rate: Late aircraft delays")

print(f"\n✅ ML READINESS:")
print("✓ Airport-specific predictions possible")
print("✓ Seasonal patterns captured")
print("✓ Delay causes quantified")
print("✓ Ready for encoding categorical features")
print("✓ Optimized for both classification and regression")


=== ML-OPTIMIZED DATASET SAVED ===
✓ ML-ready dataset saved to: ../dataset/southwest_airlines_ml_ready.csv
✓ Features: 14 (reduced from 32)
✓ Samples: 11109

=== FINAL ML FEATURE SET ===
🎯 Target Variables:
   - delay_rate: Primary prediction target
   - cancellation_rate: Secondary prediction target

📅 Temporal Features:
   - year: Long-term trends
   - month: Seasonal patterns
   - season: Categorical seasonal grouping

🏢 Airport Features:
   - airport: Specific airport (categorical)
   - airport_size: Airport size category (Small/Medium/Large)

📊 Operational Features:
   - arr_flights: Flight volume (context)
   - avg_delay_time: Average delay intensity

🔍 Delay Cause Features:
   - carrier_ct_rate: Carrier-caused delays
   - weather_ct_rate: Weather-caused delays
   - nas_ct_rate: National Air System delays
   - security_ct_rate: Security delays
   - late_aircraft_ct_rate: Late aircraft delays

✅ ML READINESS:
✓ Airport-specific predictions possible
✓ Seasonal patterns captured
✓ D

In [13]:
# Quick analysis of airport-specific patterns
print("=== AIRPORT-SPECIFIC DELAY PATTERNS ===")

# Top 10 airports by delay rate
top_delay_airports = df_ml_with_airport.groupby('airport')['delay_rate'].mean().sort_values(ascending=False).head(10)
print("Top 10 airports by average delay rate:")
for i, (airport, rate) in enumerate(top_delay_airports.items(), 1):
    print(f"  {i:2d}. {airport}: {rate:.2%}")

print(f"\nAirport size vs delay rate:")
size_delay = df_ml_with_airport.groupby('airport_size')['delay_rate'].mean().sort_values(ascending=False)
for size, rate in size_delay.items():
    print(f"  {size}: {rate:.2%}")

print(f"\nSeasonal patterns by airport size:")
seasonal_patterns = df_ml_with_airport.groupby(['airport_size', 'season'])['delay_rate'].mean().unstack()
print(seasonal_patterns.round(3))

print(f"\n=== ML USE CASES ENABLED ===")
print("🎯 Prediction scenarios:")
print("   - 'What's the delay probability for LAX in December?'")
print("   - 'Which airports have highest delay risk in summer?'")
print("   - 'How do weather delays vary by airport and season?'")
print("   - 'Predict delay rate for any airport in any month'")

print(f"\n📈 Model types possible:")
print("   - Classification: High/Medium/Low delay risk")
print("   - Regression: Exact delay rate prediction")
print("   - Time series: Seasonal trend analysis")
print("   - Clustering: Airport similarity grouping")


=== AIRPORT-SPECIFIC DELAY PATTERNS ===
Top 10 airports by average delay rate:
   1. EWR: 30.78%
   2. FAT: 29.14%
   3. ORD: 27.86%
   4. SFO: 27.50%
   5. SYR: 27.47%
   6. COS: 26.10%
   7. MIA: 25.45%
   8. BZN: 25.40%
   9. JAN: 25.16%
  10. MYR: 24.20%

Airport size vs delay rate:
  Medium: 21.32%
  Small: 21.10%
  Large: 19.50%

Seasonal patterns by airport size:
season         Fall  Spring  Summer  Winter
airport_size                               
Large         0.157   0.190   0.235   0.197
Medium        0.169   0.205   0.270   0.206
Small         0.165   0.205   0.270   0.200

=== ML USE CASES ENABLED ===
🎯 Prediction scenarios:
   - 'What's the delay probability for LAX in December?'
   - 'Which airports have highest delay risk in summer?'
   - 'How do weather delays vary by airport and season?'
   - 'Predict delay rate for any airport in any month'

📈 Model types possible:
   - Classification: High/Medium/Low delay risk
   - Regression: Exact delay rate prediction
   - Time